# 05 (Legacy) - Statistical Inference, Noise Robustness & Exploratory Figures Archive

**Archived Analysis, Inferential Testing & Exploratory Figures**

This notebook archives non-parametric statistical hypothesis testing, the noise robustness & landscape fragility suite, and exploratory figures developed during the thesis benchmarking suite:
1. **Section 1: Environment Setup & Data Ingestion**: Setup, paths, trace ingestion, and AUC-ECDF evaluation matrix preparation.
2. **Section 2: Non-Parametric Hypothesis Testing & Inferential Figures**:
   - **Figure 1 (RQ1)**: Benchmark Problem Difficulty Shift Under Stochastic Noise (Interactive).
   - **Omnibus Kruskal-Wallis & Pairwise Wilcoxon FDR Tests**: Multiplicity-adjusted hypothesis testing ($\alpha=0.05$) 
   - **Figure 4**: Vargha-Delaney Effect Size ({12}$) Pairwise Stochastic Dominance Heatmap.
   - **Figure 7**: Global Pairwise Win / Tie / Loss Tournament Ranking.
3. **Section 3: Noise Robustness & Landscape Fragility Suite**:
   - **Figure 5**: Landscape Fragility Matrix (Clean → Noisy Performance Degradation).
   - **Figure 6**: Cross-Environment Multi-Noise Robustness & Drop Profiles.
   - **Figure 9C**: Cross-Environment Noise Robustness & Performance Retention Profile.
4. **Section 4: Supplementary Architectural & ECDF Ablations**:
   - **Figure 3**: Prompt Strategy & Model Scale Multi-Bar Ablation.
   - **Figure 9B**: Empirical Runtime ECDF Performance by Dimension.
5. **Section 5: Algorithmic Failure Analysis & Error Categorization**:
   - **Figure 10**: Multi-Tier Failure Breakdown & Diagnostics Suite.

*(Note: The primary thesis publication figures are maintained in [](05_analysis.ipynb)).*

---
## Section 1: Environment Setup & Data Ingestion

### 1.1 Environment Setup, Paths & Plotly Theme Initialization
Initializes directories, Plotly typography, and statistical evaluation services.

In [1]:
%load_ext autoreload
%autoreload 2

# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from collections import defaultdict
import colorsys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shared.config import RESULTS_DIR
from shared.database.engine import create_db_session_factory
from benchmarking.infra.storage import SQLiteSynthesisReadRepository
from benchmarking.infra.io.trace_repository import IOHTraceReader
from benchmarking.application.statistical_service import StatisticalEvaluationService
from benchmarking.domain.enums import BBOBFunction
from benchmarking.domain import EvaluationCondition, EvaluationDataset, RunTrace
from benchmarking.domain.services.resolvers import (
    resolve_canonical_model_slug,
    resolve_folder_solver_name,
)
# ─── Thesis Visualization Palette & Dynamic Styling Engine ────────────────────
FONT_FAMILY = "Inter, -apple-system, BlinkMacSystemFont, Arial, sans-serif"

STRATEGY_COLOR_ARCHETYPES = {
    "guided": {"large": "#38BDF8", "small": "#38BDF8", "base": "#38BDF8"},
    "thinking": {"large": "#34D399", "small": "#34D399", "base": "#34D399"},
    "vectorization": {"large": "#F87171", "small": "#F87171", "base": "#F87171"},
    "baseline": {"large": "#FBBF24", "small": "#FBBF24", "base": "#FBBF24"},
}
STRATEGY_PALETTE = {k: v["base"] for k, v in STRATEGY_COLOR_ARCHETYPES.items()}

CLASSICAL_SOLVERS_STYLE = {
    "cma-es": {"color": "#334155", "dash": "dash", "width": 2.2, "name": "CMA-ES"},
    "pso": {"color": "#0D9488", "dash": "dashdot", "width": 2.2, "name": "PSO"},
    "de": {"color": "#7C3AED", "dash": "dot", "width": 2.2, "name": "DE"},
}

REGIME_PALETTE = {
    "clean": {"color": "#38BDF8", "border": "rgba(15, 23, 42, 0.4)", "name": "Clean (σ=0.0)", "pattern": None},
    "noisy": {"color": "#FB923C", "border": "rgba(124, 45, 18, 0.4)", "name": "Noisy (σ=0.05)", "pattern": {"shape": "/", "fillmode": "replace", "fgcolor": "#FFFFFF", "fgopacity": 0.35, "size": 6}},
}

DIMENSION_PALETTE_CLEAN = {2: "#BAE6FD", 3: "#7DD3FC", 5: "#38BDF8", 10: "#0284C7"}
DIMENSION_PALETTE_NOISY = {2: "#FED7AA", 3: "#FDBA74", 5: "#FB923C", 10: "#F97316"}
MODEL_SCALE_PALETTE = {"Qwen2.5-Coder-3B": "#BAE6FD", "Qwen2.5-Coder-7B": "#38BDF8", "Qwen2.5-Coder-14B": "#0284C7"}


def hsl_to_hex(h: float, s: float, lightness: float) -> str:
    r, g, b = colorsys.hls_to_rgb(h, lightness, s)
    return "#{:02X}{:02X}{:02X}".format(
        int(round(max(0.0, min(1.0, r)) * 255)),
        int(round(max(0.0, min(1.0, g)) * 255)),
        int(round(max(0.0, min(1.0, b)) * 255)),
    )


_DYNAMIC_STYLE_CACHE = {}


def get_solver_line_style(solver_name: str) -> dict:
    if not solver_name:
        return {"color": "#64748B", "dash": "solid", "width": 2.0}
    s = solver_name.strip()
    s_lower = s.lower()
    if s in _DYNAMIC_STYLE_CACHE:
        return _DYNAMIC_STYLE_CACHE[s]
    if s_lower in CLASSICAL_SOLVERS_STYLE:
        res = CLASSICAL_SOLVERS_STYLE[s_lower].copy()
        _DYNAMIC_STYLE_CACHE[s] = res
        return res
    if " / " in s:
        import re
        model_part, strat_part = s.split(" / ", 1)
        strat_key = strat_part.strip().lower()
        size_match = re.search(r"(\d+(?:\.\d+)?)\s*[bB]", model_part)
        is_large = float(size_match.group(1)) >= 14.0 if size_match else True
        scale_key = "large" if is_large else "small"
        line_width = 2.5 if is_large else 1.8
        if strat_key in STRATEGY_COLOR_ARCHETYPES:
            hex_color = STRATEGY_COLOR_ARCHETYPES[strat_key][scale_key]
        else:
            base_hue = (abs(hash(strat_key)) * 0.618033988749895) % 1.0
            hex_color = hsl_to_hex(base_hue, s=0.85, lightness=0.45 if is_large else 0.62)
        res = {"color": hex_color, "dash": "solid", "width": line_width}
        _DYNAMIC_STYLE_CACHE[s] = res
        return res
    hue = (abs(hash(s_lower)) * 0.618033988749895) % 1.0
    hex_color = hsl_to_hex(hue, s=0.75, lightness=0.50)
    res = {"color": hex_color, "dash": "dash", "width": 2.0}
    _DYNAMIC_STYLE_CACHE[s] = res
    return res


def get_solver_color(solver_name: str) -> str:
    return get_solver_line_style(solver_name)["color"]


def get_rgba_fill(hex_color: str, opacity: float = 0.12) -> str:
    if hex_color.startswith("#") and len(hex_color) == 7:
        r = int(hex_color[1:3], 16)
        g = int(hex_color[3:5], 16)
        b = int(hex_color[5:7], 16)
        return f"rgba({r}, {g}, {b}, {opacity})"
    return f"rgba(100, 116, 139, {opacity})"


def get_dimension_color(dim: int, is_noisy: bool = False) -> str:
    if is_noisy:
        return DIMENSION_PALETTE_NOISY.get(dim, "#FB923C")
    return DIMENSION_PALETTE_CLEAN.get(dim, "#3B82F6")


def build_dynamic_solver_palette(solvers) -> dict:
    return {s: get_solver_color(s) for s in solvers}


class DynamicSolverPalette(dict):
    def __getitem__(self, key: str) -> str:
        return get_solver_color(key)

    def get(self, key, default=None):
        if not key:
            return default or "#64748B"
        return get_solver_color(str(key))


SOLVER_PALETTE = DynamicSolverPalette()
SOLVER_LINE_STYLES = {}

session_factory = create_db_session_factory()
sqlite_repo = SQLiteSynthesisReadRepository(session_factory)
trace_reader = IOHTraceReader()
service = StatisticalEvaluationService(sqlite_repo=sqlite_repo, trace_repo=trace_reader)

# ── 1. Structured Results Subdirectories ─────────────────────────────────
# Output paths (legacy exploratory viewing)
MAIN_RESULTS_DIR  = RESULTS_DIR / "figures" / "06_thesis_summary"
CROSS_EVAL_DIR    = RESULTS_DIR / "figures" / "05_cross_evaluation"
PROFILES_DIR      = RESULTS_DIR / "figures" / "02_explicit"
REPORTS_DIR       = RESULTS_DIR / "reports"
EVALUATIONS_DIR   = RESULTS_DIR / "ioh_traces"

FILTER_DIMS = None
FILTER_PROBLEMS = None

def get_noise_color(noise_std: float, all_stds: list[float] | None = None) -> str:
    """Returns an eye-friendly soft color for any given noise level."""
    if noise_std == 0.0:
        return "#38BDF8"  # Soft sky blue for clean
    if all_stds is None or len(all_stds) <= 2:
        if np.isclose(noise_std, 0.05):
            return "#FB923C"  # Soft amber / peach
        elif np.isclose(noise_std, 0.1):
            return "#F87171"  # Soft coral
        elif np.isclose(noise_std, 0.2):
            return "#C084FC"  # Soft lavender
        return "#FB923C"
    
    noisy_stds = sorted([s for s in all_stds if s > 0.0])
    if not noisy_stds:
        return "#FB923C"
    idx = noisy_stds.index(noise_std) if noise_std in noisy_stds else 0
    palette = ["#FDBA74", "#FB923C", "#F87171", "#FB7185", "#E879F9", "#C084FC", "#A78BFA"]
    return palette[idx % len(palette)]
FILTER_NOISE_STDS = None

print("✅ Statistical service and unified dynamic thesis visualization palette initialized.")




✅ Statistical service and unified dynamic thesis visualization palette initialized.


### 1.2 Ingest Benchmark Traces & Synthesis Database Records
Loads all completed evolutionary synthesis experiments from SQLite (`data/db.sqlite3`) and empirical evaluation traces ($N=20$) from `results/ioh_traces/`.

In [2]:
# Ingest benchmark traces and synthesis records
df_exp, df_iter = service.get_synthesis_dataframes()
all_benchmark_data = service.load_evaluation_traces(
    dims=FILTER_DIMS,
    problems=FILTER_PROBLEMS,
    noise_stds=FILTER_NOISE_STDS,
    solver_resolver=resolve_folder_solver_name,
)

if not all_benchmark_data:
    raise RuntimeError(f'No benchmark traces found in {EVALUATIONS_DIR}!')

all_dims = all_benchmark_data.dims
all_noise_stds = all_benchmark_data.noise_stds
clean_std = 0.0 if 0.0 in all_noise_stds else (all_noise_stds[0] if all_noise_stds else 0.0)
noisy_std = next((n for n in all_noise_stds if n > 0.0), all_noise_stds[-1] if all_noise_stds else 0.05)
PROBLEM_IDS = all_benchmark_data.problem_ids

DISCOVERED_SOLVERS = all_benchmark_data.solvers
SOLVER_PALETTE = build_dynamic_solver_palette(DISCOVERED_SOLVERS)

MODELS_TO_SOLVERS = defaultdict(list)
for s in DISCOVERED_SOLVERS:
    if ' / ' in s:
        MODELS_TO_SOLVERS[s.split(' / ')[0]].append(s)

LLM_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' in s]
CLASSICAL_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' not in s]
ALL_SOLVERS_ORDER = LLM_SOLVERS_ORDER + CLASSICAL_SOLVERS_ORDER

print(f'📦 Loaded {len(df_exp)} experiments and {len(all_benchmark_data)} problem conditions.')
print(f'🎯 Problems: {PROBLEM_IDS} | Dimensions: {all_dims} | Solvers: {DISCOVERED_SOLVERS}')


📦 Loaded 632 experiments and 40 problem conditions.
🎯 Problems: [1, 8, 11, 15, 21] | Dimensions: [2, 3, 5, 10] | Solvers: ['CMA-ES', 'DE', 'PSO', 'Qwen2.5-Coder-14B / baseline', 'Qwen2.5-Coder-14B / baseline (noise-adapted)', 'Qwen2.5-Coder-14B / guided', 'Qwen2.5-Coder-14B / guided (noise-adapted)', 'Qwen2.5-Coder-14B / thinking', 'Qwen2.5-Coder-14B / thinking (noise-adapted)', 'Qwen2.5-Coder-14B / vectorization', 'Qwen2.5-Coder-14B / vectorization (noise-adapted)', 'Qwen2.5-Coder-3B / baseline', 'Qwen2.5-Coder-3B / baseline (noise-adapted)', 'Qwen2.5-Coder-3B / guided', 'Qwen2.5-Coder-3B / guided (noise-adapted)', 'Qwen2.5-Coder-3B / thinking', 'Qwen2.5-Coder-3B / thinking (noise-adapted)', 'Qwen2.5-Coder-3B / vectorization', 'Qwen2.5-Coder-3B / vectorization (noise-adapted)', 'Qwen2.5-Coder-7B / baseline', 'Qwen2.5-Coder-7B / baseline (noise-adapted)', 'Qwen2.5-Coder-7B / guided', 'Qwen2.5-Coder-7B / guided (noise-adapted)', 'Qwen2.5-Coder-7B / thinking', 'Qwen2.5-Coder-7B / thinkin

### 1.3 Compute 51-Target Adaptive Targets & AUC-ECDF Evaluation Matrix
Computes the adaptive precision target grid and evaluates the global AUC-ECDF matrix required by Figures 9B and 9C.

In [7]:
# ── Compute 51-Target Adaptive Targets & AUC-ECDF Matrix ───────────────────────
# Computes the AUC-ECDF performance matrix required by downstream publication figures (Figures 9C, 9D, 9E)
targets_dict = {n_std: service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std) for n_std in all_noise_stds}
df_all = service.compute_auc_ecdf_matrix(all_benchmark_data, ALL_SOLVERS_ORDER, targets=targets_dict, group_by="condition")
df_all = df_all[~df_all["Solver"].str.contains(r"\(noise-adapted\)", regex=True)]
solver_overall_auc = df_all.groupby("Solver")["AUC-ECDF (%)"].mean().sort_values(ascending=False)
sorted_solvers_auc = solver_overall_auc.index.tolist()

print(f"✅ Computed AUC-ECDF matrix for {len(df_all)} conditions across {len(sorted_solvers_auc)} solvers.")

✅ Computed AUC-ECDF matrix for 586 conditions across 15 solvers.


---
## Section 2: Non-Parametric Hypothesis Testing & Inferential Figures

### 2.0 Figure 1 (RQ1): Benchmark Difficulty Shift under Stochastic Noise
**Research Question 1**: *Does the stochastic Gaussian noise extension systematically elevate optimization difficulty across BBOB problem landscapes?*
Generates grouped bar charts comparing final error distributions in clean ($\sigma=0.0$) vs. noisy ($\sigma > 0$) environments per dimension.

In [ ]:
# ── THESIS Figure 1: Benchmark Difficulty under Noise Extension ──────────────
for dim in all_dims:
    benchmark_data_standard = {
        c: {s: runs for s, runs in s_dict.items() if "(noise-adapted)" not in s}
        for c, s_dict in all_benchmark_data.items()
    }
    clean_meds, noisy_meds, p_labels = service.compute_validation_medians(
        benchmark_data_standard, dim, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        name=f"Deterministic (σ={clean_std})",
        x=p_labels,
        y=np.maximum(clean_meds, 1e-16),
        marker=dict(color=REGIME_PALETTE["clean"]["color"], line=dict(color=REGIME_PALETTE["clean"]["border"], width=1.0))
    ))
    fig1.add_trace(go.Bar(
        name=f"Noisy Stochastic (σ={noisy_std})",
        x=p_labels,
        y=np.maximum(noisy_meds, 1e-16),
        marker=dict(
            color=REGIME_PALETTE["noisy"]["color"],
            pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.4, size=8),
            line=dict(color=REGIME_PALETTE["noisy"]["border"], width=1.0)
        )
    ))
    fig1.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 1: Benchmark Problem Difficulty under Stochastic Noise — {dim}D</b><br><span style='font-size:13px;color:#475569;font-weight:normal;'>Median Terminal Optimization Error (Δy) Across All Solvers by BBOB Landscape Class</span>",
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
            x=0.02, y=0.96
        ),
        xaxis=dict(
            title="<b>BBOB Landscape Class</b>",
            title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
            tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B")
        ),
        yaxis=dict(
            type="log",
            title="<b>Median Final Error log₁₀(Δy)</b>",
            title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
            tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"),
            range=[-16, 4],
            showgrid=True, gridwidth=1, gridcolor="#F1F5F9"
        ),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1160, height=620,
        margin=dict(l=85, r=40, t=110, b=110),
        legend=dict(
            orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
            font=dict(size=14, family=FONT_FAMILY)
        )
    )
    fig1.show()


### 2.1 Omnibus Kruskal-Wallis, Pairwise FDR Tests & Synthesis Transfer Correlation
Executes non-parametric statistical hypothesis testing across all evaluated conditions, applies Benjamini-Hochberg False Discovery Rate (FDR) control ($lpha = 0.05$)`results/reports/comprehensive_master_report.md`.

In [3]:
# ── 1. Omnibus Kruskal-Wallis & Pairwise FDR Tests ─────────────────────────
df_omnibus = service.run_omnibus_kruskal(all_benchmark_data)
df_pairwise = service.run_pairwise_fdr(all_benchmark_data, alpha=0.05)
r_val, p_val = service.compute_synthesis_transfer_correlation(df_exp)

print(f'✅ Omnibus Tests: {len(df_omnibus)} rows ({len(df_omnibus[df_omnibus["Significant"] == "Yes"])} significant)')
print(f'✅ Pairwise FDR Tests: {len(df_pairwise)} rows ({len(df_pairwise[df_pairwise["Significant (FDR)"]])} significant)')
print(f'✅ Synthesis Transfer Correlation: r = {r_val:.3f} (p = {p_val:.3e})')


✅ Omnibus Tests: 40 rows (40 significant)
✅ Pairwise FDR Tests: 5623 rows (4398 significant)
✅ Synthesis Transfer Correlation: r = 0.000 (p = 1.000e+00)
🎉 Master Comprehensive Report generated: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports/comprehensive_master_report.md


### 2.2 Figure 4: Vargha-Delaney Effect Size ($A_{12}$) Dominance Heatmap
Full pairwise non-parametric effect size matrix quantifying stochastic dominance across all solver pairs.

In [11]:
# ── THESIS Figure 4: Vargha-Delaney Effect Size (A12) Heatmap ────────────────
solvers_pw, a12_matrix = service.compute_pairwise_a12_matrix(df_pairwise)
total_conditions = len(all_benchmark_data)

fig4_a12 = go.Figure(data=go.Heatmap(
    z=a12_matrix,
    x=solvers_pw,
    y=solvers_pw,
    colorscale=[[0.0, "#F87171"], [0.25, "#FCA5A5"], [0.5, "#F8FAFC"], [0.75, "#7DD3FC"], [1.0, "#38BDF8"]],
    zmid=0.5,
    zmin=0.0,
    zmax=1.0,
    text=[[f"{val:.2f}" for val in row] for row in a12_matrix],
    texttemplate="%{text}",
    textfont=dict(size=12, family=FONT_FAMILY),
    colorbar=dict(
        title="<b>Vargha-Delaney Â₁₂</b>",
        title_font=dict(size=14, family=FONT_FAMILY),
        tickfont=dict(size=12, family=FONT_FAMILY),
        tickvals=[0.0, 0.29, 0.50, 0.71, 1.0],
        ticktext=["0.0 (Dominated)", "0.29 (Small)", "0.50 (Equal)", "0.71 (Large)", "1.0 (Dominant)"],
        len=0.85
    )
))
fig4_a12.update_layout(
    template="plotly_white",
    title=dict(
        text=f"<b>Figure 4: Global Vargha-Delaney Effect Size (Â₁₂) Heatmap</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Pairwise Non-Parametric Effect Sizes Averaged Across All {total_conditions} Problem Conditions (Row vs. Column)</span>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.96
    ),
    width=1180, height=880,
    margin=dict(l=190, r=40, t=110, b=140),
    xaxis=dict(
        title="<b>Comparison Solver (Sample 2)</b>",
        title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
        tickangle=-35,
        tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B")
    ),
    yaxis=dict(
        title="<b>Reference Solver (Sample 1)</b>",
        title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
        tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
        autorange="reversed"
    )
)
# out_fig4 = EFFECT_SIZES_DIR / "fig_04_a12_heatmap.png"
# fig4_a12.write_image(str(out_fig4), scale=3)
# print("✅ Figure 4 (A12 Heatmap) generated in results/effect_sizes/")


2026-09-03 00:04:35 INFO TemporaryDirectory.cleanup() worked.
2026-09-03 00:04:35 INFO shutil.rmtree worked.
2026-09-03 00:04:36 INFO Chromium init'ed with kwargs {}
2026-09-03 00:04:36 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-03 00:04:36 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpj5kqg77s.
2026-09-03 00:04:36 INFO Opening browser.
2026-09-03 00:04:36 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpsae8z8wz.
2026-09-03 00:04:36 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpsae8z8wz
2026-09-03 00:04:37 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpj5kqg77s/index.html
2026-09-03 00:04:37 INFO Getting tab from queue (has 1)
2026-09-03 00:04:37 INFO Got E4E3
2026-09-03 00:04:38 INFO Reloading tab E4E3 before return.
2026-09-03 00:04:38 INFO Putting tab E4E3 back (queue size: 0).
2026-09-03 00:04:38 

✅ Figure 4 (A12 Heatmap) generated in results/publication/effect_sizes/


### 2.3 Figure 7: Pairwise Win / Tie / Loss Tournament Ranking
Global head-to-head tournament matrix ranking solvers by statistically significant victories across all evaluated conditions.

In [12]:
# ── THESIS Figure 7: Pairwise Win / Tie / Loss Summary Ranking ────────────────
df_pairwise_standard = df_pairwise[
    ~df_pairwise["Solver 1"].str.contains(r"\(noise-adapted\)", regex=True) &
    ~df_pairwise["Solver 2"].str.contains(r"\(noise-adapted\)", regex=True)
]
df_wins = service.compute_pairwise_win_counts(df_pairwise_standard)
total_conditions = len(all_benchmark_data)
total_comparisons = len(df_pairwise_standard)
max_wins = int(df_wins['FDR Wins'].max()) if not df_wins.empty else 100

fig7 = go.Figure(go.Bar(
    y=df_wins["Solver"],
    x=df_wins["FDR Wins"],
    orientation="h",
    marker=dict(
        color=[get_solver_color(s) for s in df_wins["Solver"]],
        line=dict(color="#0F172A", width=1.0)
    ),
    text=df_wins["FDR Wins"],
    textposition="outside",
    textfont=dict(size=13, family=FONT_FAMILY, color="#1E293B")
))
fig7.update_layout(
    template="plotly_white",
    title=dict(
        text=f"<b>Figure 7: Global Pairwise Win Summary (Mann-Whitney U with FDR Correction)</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Total Significant Head-to-Head Victories (p < 0.05) Across All {total_conditions} BBOB Problem Conditions ({total_comparisons:,} Total Comparisons)</span>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.96
    ),
    width=1160, height=640,
    margin=dict(l=220, r=60, t=110, b=80),
    xaxis=dict(
        title="<b>Statistically Significant Wins (FDR Adjusted, α=0.05)</b>",
        title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
        tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"),
        range=[0, int(max_wins * 1.15)], showgrid=True, gridcolor="#E2E8F0", gridwidth=1.2
    ),
    yaxis=dict(
        title="<b>Optimization Solver</b>",
        title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
        tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B")
    )
)
# out_fig7 = MAIN_RESULTS_DIR / "fig_07_win_tie_loss.png"
# fig7.write_image(str(out_fig7), scale=3)
print("✅ Figure 7 (Win/Tie/Loss Ranking) generated in results/main_results/")

2026-09-03 00:04:38 INFO TemporaryDirectory.cleanup() worked.
2026-09-03 00:04:38 INFO shutil.rmtree worked.
2026-09-03 00:04:38 INFO Chromium init'ed with kwargs {}
2026-09-03 00:04:38 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-03 00:04:38 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpcryded9t.
2026-09-03 00:04:38 INFO Opening browser.
2026-09-03 00:04:38 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpmi0iihvq.
2026-09-03 00:04:38 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpmi0iihvq
2026-09-03 00:04:39 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpcryded9t/index.html
2026-09-03 00:04:40 INFO Getting tab from queue (has 1)
2026-09-03 00:04:40 INFO Got 5668
2026-09-03 00:04:40 INFO Reloading tab 5668 before return.
2026-09-03 00:04:40 INFO Putting tab 5668 back (queue size: 0).
2026-09-03 00:04:40 

✅ Figure 7 (Win/Tie/Loss Ranking) generated in results/publication/main_results/


---
## Section 3: Noise Robustness & Landscape Fragility Suite

### 3.1 Figure 5: Landscape Fragility Matrix (Clean → Noisy Degradation)
Measures the performance drop ($\Delta = \text{Clean Success Rate} - \text{Noisy Success Rate}$) across solvers by individual BBOB problem functions.

In [4]:
# ── Figure 5: Landscape Fragility Matrix (Clean → Noisy Degradation) ─────────
noisy_stds_list = [n for n in all_noise_stds if n > clean_std]
if not noisy_stds_list:
    noisy_stds_list = [0.05]

for n_std in noisy_stds_list:
    for dim in all_dims:
        frag_matrix, p_labels = service.compute_fragility_matrix(
            all_benchmark_data, dim, ALL_SOLVERS_ORDER, PROBLEM_IDS, clean_std=clean_std, noisy_std=n_std
        )
        fig5 = go.Figure(data=go.Heatmap(
            z=frag_matrix,
            x=ALL_SOLVERS_ORDER,
            y=p_labels,
            colorscale=[[0.0, "#F87171"], [0.25, "#FCA5A5"], [0.5, "#F8FAFC"], [0.75, "#7DD3FC"], [1.0, "#38BDF8"]],
            zmid=0,
            text=[[f"{v:+.2f}" for v in row] for row in frag_matrix],
            texttemplate="%{text}",
            textfont=dict(size=12, family=FONT_FAMILY),
            colorbar=dict(
                title="<b>Fragility Δ</b>",
                title_font=dict(size=14, family=FONT_FAMILY),
                tickfont=dict(size=12, family=FONT_FAMILY),
                thickness=14, len=0.85
            )
        ))
        fig5.update_layout(
            template="plotly_white",
            title=dict(
                text=f"<b>Figure 5: Landscape Fragility Matrix (Clean σ={clean_std} → Noisy σ={n_std}) — {dim}D</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Performance Drop (Δ = Clean Success Rate - Noisy Success Rate) Across Solvers by BBOB Problem</span>",
                font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
                x=0.02, y=0.96
            ),
            width=1200, height=580,
            margin=dict(l=190, r=40, t=110, b=130),
            xaxis=dict(title="<b>Optimization Solver</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickangle=-30, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B")),
            yaxis=dict(title="<b>BBOB Problem Function</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"))
        )
        # fig5.write_image(str(NOISE_DIR / f"fig_05_noise_fragility_matrix_{dim}D.png"), scale=3)
        fig5.show()
        if len(noisy_stds_list) > 1:
            pass
            # fig5.write_image(str(NOISE_DIR / f"fig_05_noise_fragility_matrix_{dim}D_std_{n_std}.png"), scale=3)

print("✅ Figure 5 (Fragility Matrix) generated in results/noise_robustness/")


2026-09-02 23:55:50 INFO Chromium init'ed with kwargs {}
2026-09-02 23:55:50 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-02 23:55:50 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkwaalan1.
2026-09-02 23:55:50 INFO Opening browser.
2026-09-02 23:55:50 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpwcpzvtuv.
2026-09-02 23:55:50 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpwcpzvtuv
2026-09-02 23:55:53 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkwaalan1/index.html
2026-09-02 23:55:54 INFO Getting tab from queue (has 1)
2026-09-02 23:55:54 INFO Got 6480
2026-09-02 23:55:54 INFO Reloading tab 6480 before return.
2026-09-02 23:55:55 INFO Putting tab 6480 back (queue size: 0).
2026-09-02 23:55:55 INFO Waiting for all cleanups to finish.
2026-09-02 23:55:55 INFO Exiting Kaleido.
2026-09-02 23:55:55 INFO T

✅ Figure 5 (Fragility Matrix) generated in results/publication/noise_robustness/


### 3.2 Figure 6: Cross-Environment Multi-Noise Robustness Profile
Analyzes overall target success rate retention across multiple noise levels ($\sigma = 0.0, 0.05, 0.1, 0.2$).

In [8]:
# ── THESIS Figure 6: Cross-Environment Multi-Noise Robustness & Drop Suite ────
df_multi_noise = service.compute_multi_noise_summary(
    all_benchmark_data, solvers=ALL_SOLVERS_ORDER, threshold=1e-8
)

# 1. Figure 6: Unified Multi-Noise Grouped Bar Chart Across Problem Dimensions
n_dims = len(all_dims)
subplot_titles = [f"<b>({chr(65+i)}) Dimension {dim}D Landscape</b>" for i, dim in enumerate(all_dims)]

fig6_unified = make_subplots(
    rows=n_dims, cols=1,
    subplot_titles=subplot_titles,
    vertical_spacing=0.14,
)

for r_idx, dim in enumerate(all_dims, start=1):
    df_d = df_multi_noise[df_multi_noise["Dim"] == dim]
    for n_idx, n_std in enumerate(all_noise_stds):
        df_dn = df_d[df_d["Noise Std"] == n_std]
        if df_dn.empty:
            continue
        c_color = get_noise_color(n_std, all_noise_stds)
        leg_name = f"Deterministic (σ=0.0)" if n_std == 0.0 else f"Noisy (σ={n_std})"
        pattern_dict = dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8) if n_std > 0 else None
        
        fig6_unified.add_trace(
            go.Bar(
                name=leg_name,
                x=df_dn["Solver"],
                y=df_dn["Success Rate"],
                marker=dict(color=c_color, pattern=pattern_dict, line=dict(color="#0F172A", width=0.8)),
                showlegend=(r_idx == 1),
            ),
            row=r_idx, col=1,
        )
    
    # Add drop badges for the most severe noisy level
    max_noisy_std = max([n for n in all_noise_stds if n > 0], default=0.05)
    df_max_noisy = df_d[df_d["Noise Std"] == max_noisy_std]
    for _, row in df_max_noisy.iterrows():
        s = row["Solver"]
        c_r = row["Clean Success Rate"]
        n_r = row["Success Rate"]
        rel_drop = row["Relative Drop Pct"]
        badge_text = f"<b>-Δ{rel_drop:.0f}%</b>" if rel_drop > 0 else "<b>0%</b>"
        badge_color = "#F87171" if rel_drop > 25 else "#34D399"
        fig6_unified.add_annotation(
            x=s, y=max(c_r, n_r) + 0.06, text=badge_text,
            showarrow=False, font=dict(size=10, color=badge_color, family=FONT_FAMILY),
            row=r_idx, col=1,
        )
        
    fig6_unified.update_xaxes(
        tickangle=-25, tickfont=dict(size=11, family=FONT_FAMILY, color="#1E293B"),
        showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=1,
    )
    fig6_unified.update_yaxes(
        title_text="<b>Success Rate</b>", title_font=dict(size=12, family=FONT_FAMILY, color="#0F172A"),
        tickfont=dict(size=11, family=FONT_FAMILY, color="#1E293B"), range=[0, 1.25],
        showgrid=True, gridcolor="#E2E8F0", gridwidth=1.0, row=r_idx, col=1,
    )

for anno in fig6_unified.layout.annotations:
    if anno.text and ("Dimension" in anno.text):
        anno.update(font=dict(size=14, color="#0F172A", family=FONT_FAMILY), x=0.02, xanchor="left")

fig6_unified.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 6: Cross-Environment Noise Robustness Profiles Across Problem Dimensions</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Empirical Target Precision Success Rate (Δy ≤ 10⁻⁸) Across Multiple Noise Levels with Relative Fragility Drops (-Δ%)</span>",
        font=dict(size=17, color="#0F172A", family=FONT_FAMILY), x=0.02, y=0.988,
    ),
    barmode="group", bargap=0.25, bargroupgap=0.08,
    width=1240, height=1600,
    margin=dict(l=75, r=40, t=110, b=80),
    legend=dict(
        orientation="h", yanchor="top", y=1.02, xanchor="right", x=0.98,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
        font=dict(size=12, family=FONT_FAMILY),
    ),
)
# out_unified = NOISE_DIR / "fig_06_robustness_profile.png"
# fig6_unified.write_image(str(out_unified), scale=3)
# 5. Individual Per-Dimension Figures
for dim in all_dims:
    df_d = df_multi_noise[df_multi_noise["Dim"] == dim]
    fig6_dim = go.Figure()
    for n_std in all_noise_stds:
        df_dn = df_d[df_d["Noise Std"] == n_std]
        if not df_dn.empty:
            c_color = get_noise_color(n_std, all_noise_stds)
            leg_name = f"Deterministic (σ=0.0)" if n_std == 0.0 else f"Noisy (σ={n_std})"
            pattern_dict = dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8) if n_std > 0 else None
            fig6_dim.add_trace(go.Bar(
                name=leg_name,
                x=df_dn["Solver"],
                y=df_dn["Success Rate"],
                marker=dict(color=c_color, pattern=pattern_dict, line=dict(color="#0F172A", width=1.0))
            ))
    fig6_dim.update_layout(
        template="plotly_white",
        title=dict(text=f"<b>Figure 6: Cross-Environment Noise Robustness Profile — {dim}D</b><br><sup>Empirical Target Success Rate (Δy ≤ 10⁻⁸) Across Available Noise Levels</sup>", font=dict(size=20, color="#0F172A", family=FONT_FAMILY), x=0.02, y=0.96),
        xaxis=dict(title="<b>Optimization Solver</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickangle=-30, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B")),
        yaxis=dict(title="<b>Overall Target Success Rate</b>", title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"), tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"), range=[0, 1.20], showgrid=True, gridcolor="#E2E8F0", gridwidth=1.2),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1220, height=660,
        margin=dict(l=75, r=40, t=100, b=120),
        legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5, bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1, font=dict(size=11, family=FONT_FAMILY))
    )
#     out_p = NOISE_DIR / f"fig_06_robustness_profile_{dim}D.png"
#     fig6_dim.write_image(str(out_p), scale=3)

print("✅ Figure 6 (Unified and Per-Dimension Profiles) generated in results/noise_robustness/")


2026-09-02 23:58:30 INFO TemporaryDirectory.cleanup() worked.
2026-09-02 23:58:30 INFO shutil.rmtree worked.
2026-09-02 23:58:31 INFO Chromium init'ed with kwargs {}
2026-09-02 23:58:31 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-02 23:58:31 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmps551fesx.
2026-09-02 23:58:31 INFO Opening browser.
2026-09-02 23:58:31 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6_2st54e.
2026-09-02 23:58:31 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6_2st54e
2026-09-02 23:58:32 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmps551fesx/index.html
2026-09-02 23:58:32 INFO Getting tab from queue (has 1)
2026-09-02 23:58:32 INFO Got 1BF1
2026-09-02 23:58:33 INFO Reloading tab 1BF1 before return.
2026-09-02 23:58:33 INFO Putting tab 1BF1 back (queue size: 0).
2026-09-02 23:58:33 

✅ Figure 6 (Unified and Per-Dimension Profiles) generated in results/publication/noise_robustness/


### 3.3 Figure 9C: Cross-Environment Noise Robustness & Performance Retention Profile
Computes the clean vs. noisy performance retention ratio across problem hardness classes, saved to `results/main_results/fig_09c_auc_ecdf_clean_vs_noisy.png`.

In [9]:
# ── THESIS Figure 9C: Cross-Environment Noise Robustness & Retention Profile ──
noisy_levels_for_9c = [n for n in all_noise_stds if n > clean_std]
target_noisy_9c = noisy_levels_for_9c[0] if noisy_levels_for_9c else 0.05

df_clean = df_all[df_all["Noise Std"] == clean_std].groupby("Solver")["AUC-ECDF (%)"].mean()
df_noisy = df_all[df_all["Noise Std"] == target_noisy_9c].groupby("Solver")["AUC-ECDF (%)"].mean()

df_ret = pd.DataFrame({
    "Clean AUC": df_clean,
    "Noisy AUC": df_noisy,
    "Retention Ratio (%)": (df_noisy / df_clean * 100).fillna(0.0)
}).sort_values(by="Noisy AUC", ascending=True)

fig9c = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"<b>(A) Clean Baseline (σ={clean_std})</b>", f"<b>(B) Noisy Environment (σ={target_noisy_9c}) & Retention</b>"],
    horizontal_spacing=0.12,
    shared_yaxes=True
)

fig9c.add_trace(go.Bar(
    y=df_ret.index, x=df_ret["Clean AUC"], orientation="h",
    name="Clean",
    marker=dict(color=[get_solver_color(s) for s in df_ret.index], line=dict(color="#0F172A", width=0.8)),
    text=[f"{v:.1f}%" for v in df_ret["Clean AUC"]],
    textposition="outside",
    textfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
    showlegend=False
), row=1, col=1)

fig9c.add_trace(go.Bar(
    y=df_ret.index, x=df_ret["Noisy AUC"], orientation="h",
    name="Noisy",
    marker=dict(
        color="rgba(255,255,255,0.85)",
        pattern=REGIME_PALETTE["noisy"]["pattern"],
        line=dict(color=[get_solver_color(s) for s in df_ret.index], width=1.5)
    ),
    text=[f"{v:.1f}% (<b>{r:.0f}% ret.</b>)" for v, r in zip(df_ret["Noisy AUC"], df_ret["Retention Ratio (%)"])],
    textposition="outside",
    textfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
    showlegend=False
), row=1, col=2)

for anno in fig9c.layout.annotations:
    anno.update(font=dict(size=16, color="#0F172A", family=FONT_FAMILY))

fig9c.update_layout(
    template="plotly_white",
    title=dict(
        text=f"<b>Figure 9C: Cross-Environment Noise Robustness & Performance Retention Profile</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Empirical Target Progress (AUC-ECDF %) in Clean vs. Noisy (σ={target_noisy_9c}) Regimes with Retention Ratios</span>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1380, height=700,
    margin=dict(l=220, r=90, t=110, b=70),
)
fig9c.update_xaxes(
    title_text=f"<b>Clean AUC-ECDF (%) [σ={clean_std}]</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    range=[0, 56], showgrid=True, gridcolor="#F1F5F9", row=1, col=1
)
fig9c.update_xaxes(
    title_text=f"<b>Noisy AUC-ECDF (%) [σ={target_noisy_9c}]</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    range=[0, 56], showgrid=True, gridcolor="#F1F5F9", row=1, col=2
)
fig9c.update_yaxes(tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), row=1, col=1)

# fig9c.write_image(str(MAIN_RESULTS_DIR / "fig_09c_auc_ecdf_clean_vs_noisy.png"), scale=3)
fig9c.show()
print("✅ Figure 9C (Clean vs Noisy Retention) generated in results/main_results/")


2026-09-08 01:03:19 INFO Chromium init'ed with kwargs {}
2026-09-08 01:03:19 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-08 01:03:19 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpab4ke14c.
2026-09-08 01:03:19 INFO Opening browser.
2026-09-08 01:03:19 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpamcvoxxy.
2026-09-08 01:03:19 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpamcvoxxy
2026-09-08 01:03:20 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpab4ke14c/index.html
2026-09-08 01:03:21 INFO Getting tab from queue (has 1)
2026-09-08 01:03:21 INFO Got 7521
2026-09-08 01:03:21 INFO Reloading tab 7521 before return.
2026-09-08 01:03:21 INFO Putting tab 7521 back (queue size: 0).
2026-09-08 01:03:21 INFO Waiting for all cleanups to finish.
2026-09-08 01:03:21 INFO Exiting Kaleido.
2026-09-08 01:03:21 INFO T

✅ Figure 9C (Clean vs Noisy Retention) generated in results/main_results/


---
## Section 4: Supplementary Architectural & ECDF Ablations

### 4.1 Figure 3: Prompt Strategy & Model Scale Ablation
Multi-bar comparison of AUC-ECDF grouped by prompt strategy (Baseline, Guided, Thinking, Vectorization) across model sizes.

In [10]:
# ── THESIS Figure 3: Prompt Strategy & Model Scale Ablation ──────────────────
for dim in all_dims:
    fig3 = go.Figure()
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        strat_labels, clean_rates, noisy_rates = service.compute_scaffolding_ablation(
            all_benchmark_data, dim, s_list, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
        )
        is_14b = "14B" in model_name
        color_c = "#0284C7" if is_14b else "#7DD3FC"
        color_n = "#D97706" if is_14b else "#FCD34D"
        fig3.add_trace(go.Bar(
            name=f"{model_name} (Clean)", x=strat_labels, y=clean_rates,
            marker=dict(color=color_c, line=dict(color="#0F172A", width=1.0)),
            text=[f"{v:.2f}" for v in clean_rates], textposition="outside",
            textfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        ))
        fig3.add_trace(go.Bar(
            name=f"{model_name} (Noisy)", x=strat_labels, y=noisy_rates,
            marker=dict(color=color_n, pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8), line=dict(color="#7C2D12", width=1.0)),
            text=[f"{v:.2f}" for v in noisy_rates], textposition="outside",
            textfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        ))
    fig3.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 3: Prompt Strategy & Model Scale Ablation — {dim}D</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Empirical Target Success Rate (Δy ≤ 10⁻⁸) by Prompt Scaffolding and Model Scale in Clean vs. Noisy Regimes</span>",
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY), x=0.02, y=0.96
        ),
        xaxis=dict(
            title="<b>Prompt Scaffolding Strategy</b>",
            title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
            tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B")
        ),
        yaxis=dict(
            title="<b>Target Success Rate</b>",
            title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
            tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"),
            range=[0, 1.20], showgrid=True, gridcolor="#E2E8F0", gridwidth=1.2
        ),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1200, height=640,
        margin=dict(l=75, r=40, t=100, b=90),
        legend=dict(
            orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
            font=dict(size=13, family=FONT_FAMILY)
        )
    )
#     out_p = ABLATION_DIR / f"fig_03_prompt_strategy_ablation_{dim}D.png"
#     fig3.write_image(str(out_p), scale=3)

print("✅ Figure 3 (Prompt Strategy Ablation) generated in results/ablation/")

2026-09-03 00:04:27 INFO Chromium init'ed with kwargs {}
2026-09-03 00:04:27 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-03 00:04:27 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpdlikxcqi.
2026-09-03 00:04:27 INFO Opening browser.
2026-09-03 00:04:27 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5lc3646z.
2026-09-03 00:04:27 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5lc3646z
2026-09-03 00:04:28 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpdlikxcqi/index.html
2026-09-03 00:04:29 INFO Getting tab from queue (has 1)
2026-09-03 00:04:29 INFO Got BE88
2026-09-03 00:04:29 INFO Reloading tab BE88 before return.
2026-09-03 00:04:29 INFO Putting tab BE88 back (queue size: 0).
2026-09-03 00:04:29 INFO Waiting for all cleanups to finish.
2026-09-03 00:04:29 INFO Exiting Kaleido.
2026-09-03 00:04:29 INFO T

✅ Figure 3 (Prompt Strategy Ablation) generated in results/publication/ablation/


### 4.2 Figure 9B: Empirical Runtime ECDF Performance by Dimension
Renders empirical cumulative distribution functions (ECDF) over evaluation budgets across problem dimensions.

In [13]:
# ── THESIS Figure 9B: Empirical Runtime ECDF Performance by Dimension ────────
targets_dict = {n_std: service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std) for n_std in all_noise_stds}

df_all = service.compute_auc_ecdf_matrix(all_benchmark_data, ALL_SOLVERS_ORDER, targets=targets_dict, group_by="condition")
df_all = df_all[~df_all["Solver"].str.contains(r"\(noise-adapted\)", regex=True)]
solver_overall_auc = df_all.groupby("Solver")["AUC-ECDF (%)"].mean().sort_values(ascending=False)
sorted_solvers_auc = solver_overall_auc.index.tolist()

fig9b = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"<b>({chr(65+i)}) Dimension {dim}D Landscape</b>" for i, dim in enumerate(all_dims)],
    vertical_spacing=0.18, horizontal_spacing=0.10
)

for idx, dim in enumerate(all_dims):
    r_idx, c_idx = (idx // 2) + 1, (idx % 2) + 1
    df_dim = df_all[df_all["Dim"] == dim]
    
    for n_idx, n_std in enumerate(all_noise_stds):
        df_cond = df_dim[df_dim["Noise Std"] == n_std]
        if df_cond.empty:
            continue
        c_color = get_noise_color(n_std, all_noise_stds)
        leg_name = f"Deterministic (σ=0.0)" if n_std == 0.0 else f"Noisy (σ={n_std})"
        pattern_dict = dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8) if n_std > 0 else None
        
        fig9b.add_trace(go.Bar(
            name=leg_name,
            x=df_cond["Solver"],
            y=df_cond["AUC-ECDF (%)"],
            marker=dict(color=c_color, pattern=pattern_dict, line=dict(color="#0F172A", width=0.8)),
            showlegend=(idx == 0)
        ), row=r_idx, col=c_idx)

    fig9b.update_xaxes(
        tickangle=-30, tickfont=dict(size=10, family=FONT_FAMILY, color="#1E293B"),
        showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx
    )
    fig9b.update_yaxes(
        title_text="<b>AUC-ECDF (%)</b>", title_font=dict(size=12, family=FONT_FAMILY, color="#0F172A"),
        tickfont=dict(size=11, family=FONT_FAMILY, color="#1E293B"), range=[0, 48],
        showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx
    )

for anno in fig9b.layout.annotations:
    anno.update(font=dict(size=15, color="#0F172A", family=FONT_FAMILY))

fig9b.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9B: Empirical Runtime ECDF Performance by Problem Dimension</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Cumulative Target Precision Progress (AUC-ECDF %) Across Problem Search Dimensions</span>",
        font=dict(size=18, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.98
    ),
    barmode="group",
    width=1340, height=860,
    margin=dict(l=70, r=40, t=110, b=120),
    legend=dict(
        orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
        font=dict(size=12, family=FONT_FAMILY)
    )
)
# out_9b = MAIN_RESULTS_DIR / "fig_09b_auc_ecdf_by_dimension.png"
# fig9b.write_image(str(out_9b), scale=3)
# fig9b.write_image(str(MAIN_RESULTS_DIR / "fig_09b_auc_ecdf_by_dim.png"), scale=3)
print("✅ Figure 9B (AUC-ECDF by Dimension) generated in results/main_results/")


2026-09-03 00:04:41 INFO Chromium init'ed with kwargs {}
2026-09-03 00:04:41 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-03 00:04:41 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5bwhh47z.
2026-09-03 00:04:41 INFO Opening browser.
2026-09-03 00:04:41 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkgm5zu03.
2026-09-03 00:04:41 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkgm5zu03
IOStream.flush timed out
2026-09-03 00:04:41 INFO TemporaryDirectory.cleanup() worked.
2026-09-03 00:04:51 INFO shutil.rmtree worked.
2026-09-03 00:04:52 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp5bwhh47z/index.html
2026-09-03 00:04:52 INFO Getting tab from queue (has 1)
2026-09-03 00:04:52 INFO Got 8DF6
2026-09-03 00:04:53 INFO Reloading tab 8DF6 before return.
2026-09-03 00:04:53 INFO Putting tab 8DF6 back (queue size:

✅ Figure 9B (AUC-ECDF by Dimension) generated in results/publication/main_results/


---
## Section 5: Algorithmic Failure Diagnostics Archive

### 5.1 Figure 10: Multi-Tier Algorithmic Outcome Breakdown & Failure Diagnostics Archive (Figures 10A, 10B, 10C, 10D)
Quantifies algorithmic convergence outcomes into four empirical precision tiers:
1. **High Precision Success** ($\Delta y \le 10^{-8}$)
2. **Medium Precision** (^{-8} < \Delta y \le 10^{-2}$)
3. **Low Precision / Stagnation** (^{-2} < \Delta y \le 10^{2}$)
4. **Severe Divergence** ($\Delta y > 10^{2}$ or Timeout)

Archives:
- **Figure 10A**: Overall Convergence Tier Outcome Breakdown ()
- **Figure 10B**: Failure Rate Heatmap Across Problem Topologies ()
- **Figure 10C**: Algorithmic Outcome Breakdown Partitioned by Dimension ()
- **Figure 10D**: Topology Heatmap Matrix Partitioned by Dimension ()


In [ ]:
# ── THESIS Figure 10: Multi-Tier Failure Breakdown & Partitioned Failure Matrices ──
tiers_df = service.compute_convergence_tiers(all_benchmark_data)
tiers_df["Canonical Solver"] = tiers_df["Solver"].str.replace(r" \(noise-adapted\)", "", regex=True)

TIER_ORDER = [
    'High Precision (Δy ≤ 10⁻⁸)',
    'Moderate Convergence (10⁻⁸ < Δy ≤ 10⁻²)',
    'Minor Progress (10⁻² < Δy ≤ 1.0)',
    'Severe Stagnation / Failure (Δy > 1.0)'
]

TIER_COLORS = {
    'High Precision (Δy ≤ 10⁻⁸)': '#38BDF8',           # Soft Sky Blue
    'Moderate Convergence (10⁻⁸ < Δy ≤ 10⁻²)': '#34D399', # Soft Mint
    'Minor Progress (10⁻² < Δy ≤ 1.0)': '#FBBF24',         # Soft Amber Gold
    'Severe Stagnation / Failure (Δy > 1.0)': '#F87171'    # Soft Coral
}

soft_colorscale = [
    [0.0, '#F8FAFC'],
    [0.2, '#FEF3C7'],
    [0.5, '#FDE68A'],
    [0.75, '#FCA5A5'],
    [1.0, '#F87171']
]

# ── 1. Figure 10A: Overall Outcome Breakdown ───────────────────────────────
solvers_order = (
    tiers_df[tiers_df['Tier'] == 'High Precision (Δy ≤ 10⁻⁸)']
    .groupby('Canonical Solver').size()
    / tiers_df.groupby('Canonical Solver').size()
).fillna(0.0).sort_values(ascending=True).index.tolist()

fig10a = go.Figure()
for tier in TIER_ORDER:
    tier_counts = tiers_df[tiers_df['Tier'] == tier].groupby('Canonical Solver').size()
    total_counts = tiers_df.groupby('Canonical Solver').size()
    tier_pcts = [(tier_counts.get(s, 0) / total_counts.get(s, 1)) * 100.0 for s in solvers_order]
    fig10a.add_trace(go.Bar(
        y=solvers_order,
        x=tier_pcts,
        name=tier,
        orientation='h',
        marker=dict(color=TIER_COLORS[tier], line=dict(color='rgba(15, 23, 42, 0.3)', width=0.8)),
        text=[f'{p:.1f}%' if p >= 5.0 else '' for p in tier_pcts],
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(size=11, family=FONT_FAMILY, color='#0F172A')
    ))

h_10a = max(720, len(solvers_order) * 28 + 180)
fig10a.update_layout(
    template='plotly_white',
    barmode='stack',
    title=dict(
        text='<b>Figure 10A: Multi-Tier Algorithmic Outcome & Failure Breakdown Across All Conditions</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Distribution of Convergence Tiers (Target Hits vs. Stagnation) Across All Evaluated Black-Box Optimizers</span>',
        font=dict(size=18, color='#0F172A', family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1380, height=h_10a,
    margin=dict(l=220, r=40, t=110, b=90),
    legend=dict(
        orientation='h', yanchor='top', y=-0.10, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#E2E8F0', borderwidth=1,
        font=dict(size=12, family=FONT_FAMILY)
    ),
    xaxis=dict(
        title='<b>Proportion of Evaluation Runs (%)</b>',
        title_font=dict(size=14, family=FONT_FAMILY, color='#0F172A'),
        tickfont=dict(size=12, family=FONT_FAMILY, color='#1E293B'),
        range=[0, 100], showgrid=True, gridcolor='#F1F5F9'
    ),
    yaxis=dict(tickfont=dict(size=12, family=FONT_FAMILY, color='#1E293B'))
)

fig10a.show()

# ── 2. Figure 10B: Failure Matrix Across BBOB Classes ───────────────────────
prob_ids = [p for p in [1, 8, 11, 15, 21] if p in PROBLEM_IDS]
prob_labels = [f'{BBOBFunction.get_name(p)}<br>(f{p})' for p in prob_ids]

solvers_order_desc = solvers_order[::-1]
matrix_all = np.zeros((len(solvers_order_desc), len(prob_ids)))
for r, s in enumerate(solvers_order_desc):
    for c, p in enumerate(prob_ids):
        cell_data = tiers_df[(tiers_df['Canonical Solver'] == s) & (tiers_df['Problem ID'] == p)]
        if not cell_data.empty:
            fail_count = (cell_data['Best Error'] > 1.0).sum()
            matrix_all[r, c] = (fail_count / len(cell_data)) * 100.0
        else:
            matrix_all[r, c] = 0.0

fig10b = go.Figure(data=go.Heatmap(
    z=matrix_all,
    x=prob_labels,
    y=solvers_order_desc,
    colorscale=soft_colorscale,
    zmin=0, zmax=100,
    text=[[f'{v:.0f}%' for v in row] for row in matrix_all],
    texttemplate='%{text}',
    textfont=dict(size=11, family=FONT_FAMILY, color='#0F172A'),
    colorbar=dict(
        title='<b>Failure Rate (%)</b>',
        title_font=dict(size=13, family=FONT_FAMILY),
        tickfont=dict(size=12, family=FONT_FAMILY),
        len=0.85
    )
))

h_10b = max(760, len(solvers_order_desc) * 28 + 180)
fig10b.update_layout(
    template='plotly_white',
    title=dict(
        text='<b>Figure 10B: Empirical Algorithmic Failure Rate Matrix Across BBOB Topologies</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Severe Stagnation / Failure Rate (Δy > 1.0) Across All Evaluated Black-Box Optimizers and BBOB Problem Classes</span>',
        font=dict(size=18, color='#0F172A', family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1180, height=h_10b,
    margin=dict(l=220, r=40, t=110, b=90),
    xaxis=dict(tickfont=dict(size=11, family=FONT_FAMILY, color='#1E293B')),
    yaxis=dict(tickfont=dict(size=11, family=FONT_FAMILY, color='#1E293B'), autorange='reversed')
)

fig10b.show()

# ── 3. Figure 10C: Breakdown Partitioned by Dimension (2D, 3D, 5D, 10D) ─────
dims = [d for d in [2, 3, 5, 10] if d in all_dims]
n_cols = 2 if len(dims) > 1 else 1
n_rows = (len(dims) + 1) // 2
fig10c = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'<b>Dimension D = {d}</b>' for d in dims],
    horizontal_spacing=0.14, vertical_spacing=0.12,
    shared_yaxes=False
)
coords = [((i // n_cols) + 1, (i % n_cols) + 1) for i in range(len(dims))]

for idx, d in enumerate(dims):
    r_idx, c_idx = coords[idx]
    sub_d = tiers_df[tiers_df['Dim'] == d]
    d_solvers = (
        sub_d[sub_d['Tier'] == 'High Precision (Δy ≤ 10⁻⁸)']
        .groupby('Canonical Solver').size()
        / sub_d.groupby('Canonical Solver').size()
    ).fillna(0.0).sort_values(ascending=True).index.tolist()

    for tier_idx, tier in enumerate(TIER_ORDER):
        tier_counts = sub_d[sub_d['Tier'] == tier].groupby('Canonical Solver').size()
        total_counts = sub_d.groupby('Canonical Solver').size()
        tier_pcts = [(tier_counts.get(s, 0) / total_counts.get(s, 1)) * 100.0 for s in d_solvers]
        fig10c.add_trace(go.Bar(
            y=d_solvers,
            x=tier_pcts,
            name=tier,
            orientation='h',
            marker=dict(color=TIER_COLORS[tier], line=dict(color='rgba(15, 23, 42, 0.3)', width=0.8)),
            showlegend=(idx == 0),
            text=[f'{p:.0f}%' if p >= 8.0 else '' for p in tier_pcts],
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(size=10, family=FONT_FAMILY, color='#0F172A')
        ), row=r_idx, col=c_idx)

    fig10c.update_xaxes(range=[0, 100], title_text='<b>Run Percentage (%)</b>' if r_idx==n_rows else '', row=r_idx, col=c_idx)
    fig10c.update_yaxes(tickfont=dict(size=11, family=FONT_FAMILY, color='#1E293B'), row=r_idx, col=c_idx)

for anno in fig10c.layout.annotations:
    anno.update(font=dict(size=15, color='#0F172A', family=FONT_FAMILY))

fig10c.update_layout(
    template='plotly_white',
    barmode='stack',
    title=dict(
        text='<b>Figure 10C: Empirical Algorithmic Failure Breakdown Partitioned by Problem Dimension</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Distribution of Convergence Tiers and Stagnation Rates Across Search Dimensions</span>',
        font=dict(size=18, color='#0F172A', family=FONT_FAMILY),
        x=0.02, y=0.98
    ),
    width=1480, height=920,
    margin=dict(l=220, r=40, t=110, b=80),
    legend=dict(
        orientation='h', yanchor='top', y=-0.08, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#E2E8F0', borderwidth=1,
        font=dict(size=12, family=FONT_FAMILY)
    )
)

fig10c.show()


# ── THESIS Figure 10D: Failure Rate Matrix Partitioned by Dimension ────────────
tiers_df = service.compute_convergence_tiers(all_benchmark_data)
tiers_df = tiers_df[~tiers_df["Solver"].str.contains(r"\(noise-adapted\)", regex=True)]

dims = [d for d in all_dims if d in [2, 3, 5, 10]]
prob_ids = [1, 8, 11, 15, 21]
prob_labels = [f"{BBOBFunction.get_name(p)}<br>(f{p})" for p in prob_ids]
coords = [((i // 2) + 1, (i % 2) + 1) for i in range(4)]

soft_colorscale = [
    [0.0, '#F8FAFC'],
    [0.2, '#FEF3C7'],
    [0.5, '#FDE68A'],
    [0.75, '#FCA5A5'],
    [1.0, '#F87171']
]

# ── 4. Figure 10D: Failure Rate Matrix Partitioned by Dimension ────────────
fig10d = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>Dimension D = {d}</b>' for d in dims],
    horizontal_spacing=0.12, vertical_spacing=0.14,
    shared_yaxes=False
)

for idx, d in enumerate(dims):
    r_idx, c_idx = coords[idx]
    sub_d = tiers_df[tiers_df['Dim'] == d]
    
    d_solvers = (
        sub_d[sub_d['Tier'] == 'High Precision (Δy ≤ 10⁻⁸)']
        .groupby('Solver').size()
        / sub_d.groupby('Solver').size()
    ).fillna(0.0).sort_values(ascending=False).index.tolist()
    
    matrix = np.zeros((len(d_solvers), len(prob_ids)))
    for r, s in enumerate(d_solvers):
        for c, p in enumerate(prob_ids):
            cell_data = sub_d[(sub_d['Solver'] == s) & (sub_d['Problem ID'] == p)]
            if not cell_data.empty:
                fail_count = (cell_data['Best Error'] > 1.0).sum()
                matrix[r, c] = (fail_count / len(cell_data)) * 100.0
            else:
                matrix[r, c] = 0.0

    fig10d.add_trace(go.Heatmap(
        z=matrix,
        x=prob_labels,
        y=d_solvers,
        colorscale=soft_colorscale,
        zmin=0, zmax=100,
        text=[[f'{v:.0f}%' for v in row] for row in matrix],
        texttemplate='%{text}',
        textfont=dict(size=10, family=FONT_FAMILY, color='#0F172A'),
        showscale=(idx == len(dims)-1),
        colorbar=dict(
            title='<b>Failure Rate (%)</b>',
            title_font=dict(size=12, family=FONT_FAMILY),
            tickfont=dict(size=11, family=FONT_FAMILY),
            len=0.85
        ) if idx == len(dims)-1 else None
    ), row=r_idx, col=c_idx)

    fig10d.update_xaxes(tickfont=dict(size=10, family=FONT_FAMILY, color='#1E293B'), row=r_idx, col=c_idx)
    fig10d.update_yaxes(tickfont=dict(size=10, family=FONT_FAMILY, color='#1E293B'), autorange='reversed', row=r_idx, col=c_idx)

for anno in fig10d.layout.annotations:
    anno.update(font=dict(size=15, color='#0F172A', family=FONT_FAMILY))

fig10d.update_layout(
    template='plotly_white',
    title=dict(
        text='<b>Figure 10D: Empirical Algorithmic Failure Rate Matrix Partitioned by Problem Dimension</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Severe Stagnation / Failure Rate (Δy > 1.0) Across Solver Strategies and BBOB Problem Topologies</span>',
        font=dict(size=18, color='#0F172A', family=FONT_FAMILY),
        x=0.02, y=0.98
    ),
    width=1480, height=920,
    margin=dict(l=220, r=40, t=110, b=80),
)

fig10d.show()
